# Exploratory Data Analysis: MovieLens Recommender Systems

**Purpose**: A rigorous EDA of the MovieLens dataset to inform the design, evaluation, and interpretation of recommender system experiments. This analysis provides the empirical foundation for our study comparing traditional and neural recommender models under data-sparsity and cold-start conditions.

---

## Table of Contents
1. [Dataset Overview & Integrity Checks](#1)
2. [Rating Distribution Analysis](#2)
3. [User Activity Analysis — Power Law & Engagement](#3)
4. [Item Popularity Analysis — Long-Tail Distribution](#4)
5. [Temporal Analysis — Rating Trends Over Time](#5)
6. [Genre Analysis — Taxonomy, Co-occurrence, Rating Patterns](#6)
7. [Sparsity Analysis — Matrix Density & Implications](#7)
8. [Cold-Start Severity Quantification](#8)
9. [User Rating Behaviour — Bias & Variance](#9)
10. [Statistical Tests & Distributional Assumptions](#10)
11. [Summary of Key Findings for Model Design](#11)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats, sparse
from scipy.sparse import csr_matrix
from collections import Counter
from itertools import combinations
from pathlib import Path

# Publication-quality defaults
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'font.size': 12,
    'font.family': 'serif',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = sns.color_palette('Set2', 10)
sns.set_palette(PALETTE)

DATA_DIR = Path('..') / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path('..')

print(f'Data directory: {DATA_DIR.resolve()}')

<a id='1'></a>
## 1. Dataset Overview & Integrity Checks

We begin with a structural audit: schema validation, missing values, duplicate detection, and dtype verification.

In [ ]:
# Load data
ratings = pd.read_csv(DATA_DIR / 'ratings.csv')
movies = pd.read_csv(DATA_DIR / 'movies.csv')

print('=== RATINGS ===')
print(f'Shape: {ratings.shape}')
print(f'Memory: {ratings.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Dtypes:\n{ratings.dtypes}\n')
print(f'Missing values:\n{ratings.isnull().sum()}\n')
print(f'Duplicates: {ratings.duplicated(subset=["userId","movieId"]).sum()}')

print('\n=== MOVIES ===')
print(f'Shape: {movies.shape}')
print(f'Memory: {movies.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Missing values:\n{movies.isnull().sum()}')

ratings.head()

In [ ]:
# Basic statistics
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)

summary = pd.DataFrame({
    'Metric': ['Total ratings', 'Unique users', 'Unique movies',
               'Avg ratings/user', 'Avg ratings/movie',
               'Rating range', 'Rating scale'],
    'Value': [
        f'{n_ratings:,}', f'{n_users:,}', f'{n_movies:,}',
        f'{n_ratings/n_users:.1f}', f'{n_ratings/n_movies:.1f}',
        f'{ratings["rating"].min():.1f} – {ratings["rating"].max():.1f}',
        f'{ratings["rating"].nunique()} levels (0.5 increments)'
    ]
})
print(summary.to_string(index=False))

print(f'\nMatrix density: {n_ratings / (n_users * n_movies) * 100:.4f}%')
print(f'Sparsity: {(1 - n_ratings / (n_users * n_movies)) * 100:.4f}%')

<a id='2'></a>
## 2. Rating Distribution Analysis

Understanding the marginal rating distribution reveals user tendencies (positivity bias is common in recommender datasets) and informs threshold selection for binary classification metrics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Histogram
rating_counts = ratings['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, width=0.4, color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('(a) Rating Frequency Distribution')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
# Add percentage labels
for x, y in zip(rating_counts.index, rating_counts.values):
    axes[0].text(x, y + n_ratings*0.005, f'{y/n_ratings*100:.1f}%', ha='center', fontsize=8)

# (b) CDF
sorted_ratings = np.sort(ratings['rating'].values)
cdf = np.arange(1, len(sorted_ratings)+1) / len(sorted_ratings)
axes[1].plot(sorted_ratings, cdf, color=PALETTE[1], linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Median')
axes[1].axvline(3.5, color='red', linestyle='--', alpha=0.5, label='Threshold=3.5')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Cumulative Probability')
axes[1].set_title('(b) Cumulative Distribution Function')
axes[1].legend()

# (c) Box + violin
parts = axes[2].violinplot(ratings['rating'].values, positions=[1], showmeans=True, showmedians=True)
for pc in parts['bodies']:
    pc.set_facecolor(PALETTE[2])
    pc.set_alpha(0.7)
axes[2].set_ylabel('Rating')
axes[2].set_title('(c) Rating Violin Plot')
axes[2].set_xticks([1])
axes[2].set_xticklabels(['All Ratings'])

plt.suptitle('Rating Distribution Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Summary statistics
print(f'Mean: {ratings["rating"].mean():.3f}')
print(f'Median: {ratings["rating"].median():.3f}')
print(f'Std: {ratings["rating"].std():.3f}')
print(f'Skewness: {ratings["rating"].skew():.3f}')
print(f'Kurtosis: {ratings["rating"].kurtosis():.3f}')
print(f'\nFraction >= 3.5 ("liked"): {(ratings["rating"] >= 3.5).mean()*100:.1f}%')
print(f'Fraction >= 4.0: {(ratings["rating"] >= 4.0).mean()*100:.1f}%')

<a id='3'></a>
## 3. User Activity Analysis — Power Law & Engagement

User activity in recommender datasets typically follows a power law: a small fraction of users contribute the majority of ratings. This has direct implications for collaborative filtering and cold-start severity.

In [ ]:
user_activity = ratings.groupby('userId')['rating'].count().rename('n_ratings')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Distribution of ratings per user (log scale)
axes[0].hist(user_activity, bins=100, color=PALETTE[0], edgecolor='white', log=True)
axes[0].axvline(user_activity.median(), color='red', linestyle='--', label=f'Median={user_activity.median():.0f}')
axes[0].axvline(user_activity.mean(), color='blue', linestyle='--', label=f'Mean={user_activity.mean():.0f}')
axes[0].set_xlabel('Number of Ratings per User')
axes[0].set_ylabel('Number of Users (log)')
axes[0].set_title('(a) User Activity Distribution')
axes[0].legend()

# (b) Log-log plot (power law check)
sorted_activity = np.sort(user_activity.values)[::-1]
rank = np.arange(1, len(sorted_activity) + 1)
axes[1].loglog(rank, sorted_activity, '.', markersize=1, color=PALETTE[1], alpha=0.5)
axes[1].set_xlabel('Rank (log)')
axes[1].set_ylabel('Ratings Count (log)')
axes[1].set_title('(b) Zipf Plot — Power Law Check')

# Fit power law
log_rank = np.log10(rank)
log_activity = np.log10(sorted_activity)
slope, intercept, r_value, _, _ = stats.linregress(log_rank, log_activity)
fit_line = 10**(intercept + slope * log_rank)
axes[1].loglog(rank, fit_line, '--', color='red', label=f'Slope={slope:.2f}, R²={r_value**2:.3f}')
axes[1].legend()

# (c) Lorenz curve (inequality)
sorted_asc = np.sort(user_activity.values)
cumulative = np.cumsum(sorted_asc) / sorted_asc.sum()
x = np.arange(1, len(cumulative)+1) / len(cumulative)
axes[2].plot(x, cumulative, color=PALETTE[2], linewidth=2, label='Lorenz Curve')
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Equality')
axes[2].fill_between(x, x, cumulative, alpha=0.15, color=PALETTE[2])
gini = 1 - 2 * np.trapz(cumulative, x)
axes[2].set_xlabel('Cumulative Share of Users')
axes[2].set_ylabel('Cumulative Share of Ratings')
axes[2].set_title(f'(c) Lorenz Curve (Gini={gini:.3f})')
axes[2].legend()

plt.suptitle('User Activity Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Quantile analysis
quantiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print('User activity quantiles:')
for q in quantiles:
    print(f'  {q*100:5.1f}% of users rated <= {user_activity.quantile(q):.0f} movies')

# Top contributors
top_1pct = int(n_users * 0.01)
top_ratings = user_activity.nlargest(top_1pct).sum()
print(f'\nTop 1% users ({top_1pct} users) contribute {top_ratings/n_ratings*100:.1f}% of all ratings')

<a id='4'></a>
## 4. Item Popularity Analysis — Long-Tail Distribution

The long-tail phenomenon is central to recommender system design: most items receive few ratings while a handful dominate.

In [ ]:
item_popularity = ratings.groupby('movieId')['rating'].agg(['count', 'mean', 'std']).rename(
    columns={'count': 'n_ratings', 'mean': 'avg_rating', 'std': 'std_rating'}
)
item_popularity = item_popularity.merge(movies[['movieId', 'title']], left_index=True, right_on='movieId')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Long-tail
sorted_pop = item_popularity['n_ratings'].sort_values(ascending=False).values
axes[0].plot(range(len(sorted_pop)), sorted_pop, color=PALETTE[0], linewidth=0.8)
axes[0].set_xlabel('Movie Rank')
axes[0].set_ylabel('Number of Ratings')
axes[0].set_title('(a) Long-Tail Distribution')
axes[0].set_yscale('log')
# Mark the "head" (top 20% items contributing 80% ratings)
cum_pop = np.cumsum(sorted_pop) / sorted_pop.sum()
head_cutoff = np.searchsorted(cum_pop, 0.8)
axes[0].axvline(head_cutoff, color='red', linestyle='--', 
                label=f'Top {head_cutoff} movies = 80% ratings')
axes[0].legend(fontsize=8)

# (b) Ratings count vs avg rating
sc = axes[1].scatter(
    item_popularity['n_ratings'], item_popularity['avg_rating'],
    s=2, alpha=0.3, c=PALETTE[1]
)
axes[1].set_xlabel('Number of Ratings (log scale)')
axes[1].set_ylabel('Average Rating')
axes[1].set_title('(b) Popularity vs Quality')
axes[1].set_xscale('log')
# Trend line
bins = pd.cut(np.log10(item_popularity['n_ratings']+1), 20)
binned = item_popularity.groupby(bins, observed=True)['avg_rating'].mean()
bin_centers = [interval.mid for interval in binned.index]
axes[1].plot(10**np.array(bin_centers), binned.values, 'r-', linewidth=2, label='Binned mean')
axes[1].legend()

# (c) Distribution of movie rating counts
axes[2].hist(item_popularity['n_ratings'], bins=100, color=PALETTE[2], edgecolor='white', log=True)
axes[2].axvline(item_popularity['n_ratings'].median(), color='red', linestyle='--', 
                label=f'Median={item_popularity["n_ratings"].median():.0f}')
axes[2].set_xlabel('Number of Ratings per Movie')
axes[2].set_ylabel('Number of Movies (log)')
axes[2].set_title('(c) Item Rating Frequency')
axes[2].legend()

plt.suptitle('Item Popularity Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Top & bottom movies
min_ratings = 100
popular = item_popularity[item_popularity['n_ratings'] >= min_ratings]
print(f'\nTop 10 movies by average rating (min {min_ratings} ratings):')
print(popular.nlargest(10, 'avg_rating')[['title', 'avg_rating', 'n_ratings']].to_string(index=False))
print(f'\nTop 10 most-rated movies:')
print(item_popularity.nlargest(10, 'n_ratings')[['title', 'avg_rating', 'n_ratings']].to_string(index=False))

<a id='5'></a>
## 5. Temporal Analysis — Rating Trends Over Time

Temporal patterns affect evaluation validity. Rating volume, user behaviour, and genre preferences can shift over time.

In [ ]:
import re

# Extract year from movie titles
def extract_year(title):
    match = re.search(r'\((\d{4})\)', str(title))
    return int(match.group(1)) if match else None

movies['year'] = movies['title'].apply(extract_year)
movies_with_year = movies.dropna(subset=['year'])
movies_with_year['year'] = movies_with_year['year'].astype(int)

# Merge with ratings
ratings_with_year = ratings.merge(movies_with_year[['movieId', 'year']], on='movieId')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Movie release year distribution
year_counts = movies_with_year['year'].value_counts().sort_index()
axes[0].bar(year_counts.index, year_counts.values, color=PALETTE[0], width=1, edgecolor='none')
axes[0].set_xlabel('Release Year')
axes[0].set_ylabel('Number of Movies')
axes[0].set_title('(a) Movie Release Year Distribution')
axes[0].set_xlim(1900, 2020)

# (b) Average rating by decade
ratings_with_year['decade'] = (ratings_with_year['year'] // 10) * 10
decade_stats = ratings_with_year.groupby('decade')['rating'].agg(['mean', 'std', 'count'])
decade_stats = decade_stats[decade_stats['count'] >= 100]  # filter sparse decades

axes[1].errorbar(decade_stats.index, decade_stats['mean'], 
                 yerr=decade_stats['std']/np.sqrt(decade_stats['count']),
                 fmt='o-', color=PALETTE[1], capsize=3, markersize=6)
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Average Rating')
axes[1].set_title('(b) Average Rating by Decade')
axes[1].set_ylim(2.5, 4.5)

# (c) Rating volume by movie age
ratings_with_year['movie_age'] = ratings_with_year['year'].max() - ratings_with_year['year']
age_ratings = ratings_with_year.groupby('movie_age')['rating'].count()
axes[2].plot(age_ratings.index, age_ratings.values, color=PALETTE[2], linewidth=1)
axes[2].set_xlabel('Movie Age (years from newest)')
axes[2].set_ylabel('Number of Ratings')
axes[2].set_title('(c) Rating Volume by Movie Age')
axes[2].set_yscale('log')

plt.suptitle('Temporal Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f'Year range: {movies_with_year["year"].min()} – {movies_with_year["year"].max()}')
print(f'Movies without parseable year: {len(movies) - len(movies_with_year)}')

<a id='6'></a>
## 6. Genre Analysis — Taxonomy, Co-occurrence, Rating Patterns

Genre is the primary content feature in our content-based recommender. Understanding the genre taxonomy, multi-labelling structure, and genre-rating correlations is essential for TF-IDF feature engineering.

In [ ]:
# Explode genres
movies_exploded = movies_with_year.assign(
    genre=movies_with_year['genres'].str.split('|')
).explode('genre')

genre_counts = movies_exploded['genre'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Genre frequency
genre_counts.plot.barh(ax=axes[0], color=PALETTE[0], edgecolor='white')
axes[0].set_xlabel('Number of Movies')
axes[0].set_title('(a) Genre Frequency')
axes[0].invert_yaxis()

# (b) Average rating per genre
ratings_with_genre = ratings.merge(movies_exploded[['movieId', 'genre']], on='movieId')
genre_rating_stats = ratings_with_genre.groupby('genre')['rating'].agg(['mean', 'std', 'count'])
genre_rating_stats = genre_rating_stats.sort_values('mean', ascending=True)

axes[1].barh(genre_rating_stats.index, genre_rating_stats['mean'], 
             xerr=genre_rating_stats['std']/np.sqrt(genre_rating_stats['count']),
             color=PALETTE[1], edgecolor='white', capsize=2)
axes[1].set_xlabel('Average Rating')
axes[1].set_title('(b) Average Rating by Genre')
axes[1].set_xlim(2.5, 4.5)

plt.suptitle('Genre Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f'Total unique genres: {genre_counts.shape[0]}')
print(f'Average genres per movie: {movies_with_year["genres"].str.count("\\|").add(1).mean():.2f}')

In [ ]:
# Genre co-occurrence matrix
genre_list = sorted(genre_counts.index.tolist())
if '(no genres listed)' in genre_list:
    genre_list.remove('(no genres listed)')

cooccurrence = pd.DataFrame(0, index=genre_list, columns=genre_list)

for genres_str in movies_with_year['genres']:
    gs = [g for g in genres_str.split('|') if g in genre_list]
    for g1, g2 in combinations(gs, 2):
        cooccurrence.loc[g1, g2] += 1
        cooccurrence.loc[g2, g1] += 1
    for g in gs:
        cooccurrence.loc[g, g] += 1

# Normalise by diagonal (Jaccard-like)
diag = np.diag(cooccurrence.values).copy()
diag[diag == 0] = 1
normalised = cooccurrence.values / np.sqrt(np.outer(diag, diag))
np.fill_diagonal(normalised, 1)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(normalised, dtype=bool), k=1)
sns.heatmap(normalised, mask=mask, xticklabels=genre_list, yticklabels=genre_list,
            cmap='YlOrRd', vmin=0, vmax=0.5, annot=False, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Genre Co-occurrence Matrix (Normalised)', fontsize=14)
plt.tight_layout()
plt.show()

<a id='7'></a>
## 7. Sparsity Analysis — Matrix Density & Implications

The utility matrix sparsity is the fundamental challenge. We visualise the sparsity pattern and quantify its impact on different algorithmic families.

In [ ]:
# Build a sampled utility matrix for visualisation
np.random.seed(42)
sample_users = np.random.choice(ratings['userId'].unique(), min(200, n_users), replace=False)
sample_movies = np.random.choice(ratings['movieId'].unique(), min(300, n_movies), replace=False)

sample_ratings = ratings[
    ratings['userId'].isin(sample_users) & ratings['movieId'].isin(sample_movies)
]

# Build pivot
user_map = {u: i for i, u in enumerate(sorted(sample_users))}
movie_map = {m: i for i, m in enumerate(sorted(sample_movies))}

matrix = np.zeros((len(sample_users), len(sample_movies)))
for _, row in sample_ratings.iterrows():
    if row['userId'] in user_map and row['movieId'] in movie_map:
        matrix[user_map[row['userId']], movie_map[row['movieId']]] = row['rating']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (a) Sparsity pattern
axes[0].spy(matrix != 0, markersize=0.3, color=PALETTE[0], aspect='auto')
axes[0].set_xlabel('Movies (sample)')
axes[0].set_ylabel('Users (sample)')
axes[0].set_title(f'(a) Utility Matrix Sparsity Pattern\n(sample: {matrix.shape[0]}×{matrix.shape[1]})')

# (b) Density at different sample sizes
fractions = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0]
densities = []
for frac in fractions:
    n = int(n_users * frac)
    if n < 1:
        n = 1
    sub_users = np.random.choice(ratings['userId'].unique(), min(n, n_users), replace=False)
    sub = ratings[ratings['userId'].isin(sub_users)]
    sub_n_items = sub['movieId'].nunique()
    density = len(sub) / (len(sub_users) * sub_n_items) if sub_n_items > 0 else 0
    densities.append(density * 100)

axes[1].plot(fractions, densities, 'o-', color=PALETTE[1], markersize=6)
axes[1].set_xlabel('Sample Fraction of Users')
axes[1].set_ylabel('Matrix Density (%)')
axes[1].set_title('(b) Density vs Sample Size')
axes[1].set_xscale('log')

plt.suptitle('Sparsity Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print(f'Full matrix density: {n_ratings / (n_users * n_movies) * 100:.4f}%')
print(f'Sample matrix ({matrix.shape}) density: {(matrix != 0).sum() / matrix.size * 100:.2f}%')

<a id='8'></a>
## 8. Cold-Start Severity Quantification

We define cold-start thresholds and quantify how many users and items fall below them. This directly motivates our experimental scenarios.

In [ ]:
thresholds = [1, 2, 3, 5, 10, 20, 50]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# User cold-start
user_cold = []
for t in thresholds:
    pct = (user_activity <= t).mean() * 100
    user_cold.append(pct)

axes[0].bar(range(len(thresholds)), user_cold, color=PALETTE[3], edgecolor='white')
axes[0].set_xticks(range(len(thresholds)))
axes[0].set_xticklabels([f'<={t}' for t in thresholds])
axes[0].set_xlabel('Rating Count Threshold')
axes[0].set_ylabel('% of Users')
axes[0].set_title('(a) User Cold-Start Severity')
for i, v in enumerate(user_cold):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

# Item cold-start
item_cold = []
item_activity = ratings.groupby('movieId')['rating'].count()
for t in thresholds:
    pct = (item_activity <= t).mean() * 100
    item_cold.append(pct)

axes[1].bar(range(len(thresholds)), item_cold, color=PALETTE[4], edgecolor='white')
axes[1].set_xticks(range(len(thresholds)))
axes[1].set_xticklabels([f'<={t}' for t in thresholds])
axes[1].set_xlabel('Rating Count Threshold')
axes[1].set_ylabel('% of Movies')
axes[1].set_title('(b) Item Cold-Start Severity')
for i, v in enumerate(item_cold):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

plt.suptitle('Cold-Start Severity Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print('Cold-start users (<=5 ratings):  ', f'{(user_activity <= 5).sum():,} ({(user_activity <= 5).mean()*100:.1f}%)')
print('Cold-start items (<=5 ratings):  ', f'{(item_activity <= 5).sum():,} ({(item_activity <= 5).mean()*100:.1f}%)')
print(f'\nUsers with exactly 1 rating: {(user_activity == 1).sum():,} ({(user_activity == 1).mean()*100:.1f}%)')
print(f'Movies with exactly 1 rating: {(item_activity == 1).sum():,} ({(item_activity == 1).mean()*100:.1f}%)')

<a id='9'></a>
## 9. User Rating Behaviour — Bias & Variance

Users exhibit systematic biases: some are generous raters, others are strict. This heterogeneity affects collaborative filtering and motivates mean-centring in Pearson-based approaches.

In [ ]:
user_stats = ratings.groupby('userId')['rating'].agg(['mean', 'std', 'count']).rename(
    columns={'mean': 'avg_rating', 'std': 'std_rating', 'count': 'n_ratings'}
)
# Filter users with enough ratings for meaningful std
active_users = user_stats[user_stats['n_ratings'] >= 20]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) Distribution of user mean ratings
axes[0].hist(active_users['avg_rating'], bins=50, color=PALETTE[0], edgecolor='white', density=True)
axes[0].axvline(active_users['avg_rating'].mean(), color='red', linestyle='--',
                label=f'Mean={active_users["avg_rating"].mean():.2f}')
axes[0].set_xlabel('User Mean Rating')
axes[0].set_ylabel('Density')
axes[0].set_title('(a) Distribution of User Rating Bias')
axes[0].legend()

# (b) Distribution of user rating variance
axes[1].hist(active_users['std_rating'].dropna(), bins=50, color=PALETTE[1], edgecolor='white', density=True)
axes[1].axvline(active_users['std_rating'].mean(), color='red', linestyle='--',
                label=f'Mean Std={active_users["std_rating"].mean():.2f}')
axes[1].set_xlabel('User Rating Std Dev')
axes[1].set_ylabel('Density')
axes[1].set_title('(b) Rating Discrimination (Variance)')
axes[1].legend()

# (c) Mean vs Std scatter
axes[2].scatter(active_users['avg_rating'], active_users['std_rating'],
                s=3, alpha=0.3, c=PALETTE[2])
axes[2].set_xlabel('User Mean Rating')
axes[2].set_ylabel('User Std Dev')
axes[2].set_title('(c) Bias vs Discrimination')
# Add correlation
r, p = stats.pearsonr(active_users['avg_rating'], active_users['std_rating'].fillna(0))
axes[2].text(0.05, 0.95, f'r = {r:.3f}', transform=axes[2].transAxes, fontsize=11,
             verticalalignment='top')

plt.suptitle('User Rating Behaviour', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Extreme raters
generous = active_users[active_users['avg_rating'] >= 4.5]
strict = active_users[active_users['avg_rating'] <= 2.0]
print(f'"Generous" raters (mean >= 4.5): {len(generous)} ({len(generous)/len(active_users)*100:.1f}%)')
print(f'"Strict" raters (mean <= 2.0):   {len(strict)} ({len(strict)/len(active_users)*100:.1f}%)')

<a id='10'></a>
## 10. Statistical Tests & Distributional Assumptions

We test whether ratings follow common distributional assumptions and whether significant differences exist between genre ratings.

In [ ]:
# Normality test on rating distribution (sample due to test limitations)
sample_size = 5000
rating_sample = ratings['rating'].sample(sample_size, random_state=42)

# Shapiro-Wilk
stat_sw, p_sw = stats.shapiro(rating_sample)
print(f'Shapiro-Wilk test (n={sample_size}):')
print(f'  Statistic: {stat_sw:.6f}')
print(f'  p-value:   {p_sw:.2e}')
print(f'  Normal?    {"Yes" if p_sw > 0.05 else "No (reject H0)"}')

# D'Agostino-Pearson
stat_dp, p_dp = stats.normaltest(rating_sample)
print(f'\nD\'Agostino-Pearson test:')
print(f'  Statistic: {stat_dp:.6f}')
print(f'  p-value:   {p_dp:.2e}')

# Kolmogorov-Smirnov against normal
stat_ks, p_ks = stats.kstest(rating_sample, 'norm', 
                              args=(rating_sample.mean(), rating_sample.std()))
print(f'\nKolmogorov-Smirnov test (vs Normal):')
print(f'  Statistic: {stat_ks:.6f}')
print(f'  p-value:   {p_ks:.2e}')

print('\n' + '='*50)
print('ANOVA: Do genres have significantly different ratings?')
print('='*50)

# ANOVA across major genres
genre_groups = []
genre_names = []
top_genres = genre_counts.head(8).index.tolist()
for genre in top_genres:
    genre_ratings = ratings_with_genre[ratings_with_genre['genre'] == genre]['rating'].values
    if len(genre_ratings) > 100:
        genre_groups.append(genre_ratings)
        genre_names.append(genre)

f_stat, p_anova = stats.f_oneway(*genre_groups)
print(f'\nOne-way ANOVA across {len(genre_names)} genres:')
print(f'  F-statistic: {f_stat:.2f}')
print(f'  p-value:     {p_anova:.2e}')
print(f'  Significant? {"Yes" if p_anova < 0.001 else "No"}')

# Effect size (eta-squared)
all_ratings_flat = np.concatenate(genre_groups)
grand_mean = all_ratings_flat.mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in genre_groups)
ss_total = sum((g - grand_mean).var() * len(g) for g in genre_groups) + ss_between
eta_sq = ss_between / ss_total
print(f'  Effect size (η²): {eta_sq:.4f} ({"small" if eta_sq < 0.06 else "medium" if eta_sq < 0.14 else "large"})')

<a id='11'></a>
## 11. Summary of Key Findings for Model Design

| Finding | Implication for Recommender Design |
|---------|------------------------------------|
| Rating distribution is left-skewed (positivity bias) | Threshold at 3.5 captures ~50% as "liked" — reasonable binary split |
| User activity follows power law (Gini > 0.5) | Collaborative filtering will struggle for low-activity users; content-based may be preferable |
| Extreme long-tail in item popularity | Matrix completion and embedding methods must handle rare items |
| Matrix sparsity > 98% | SVT/ALS need pre-filling strategies; neural methods need regularisation |
| Significant cold-start: many users with <5 ratings | Validates our cold-start experimental scenario |
| User rating bias varies widely (mean 1.5–5.0) | Pearson correlation and mean-centring are justified |
| Genre ratings are statistically different (ANOVA p<0.001) | Content-based features carry signal; TF-IDF on genres is meaningful |
| Older movies tend to have higher ratings (survivorship bias) | Temporal splits should be considered for future work |